# 02 — Feature engineering and causal scaling

Three-way windows: **train** (fit), **selection** (2022–2023 attribution/shortlist), **final** (2024 one-shot validation). Timestamp contract: close_t → target r_{t+1}.


In [ ]:
from spy_alpha_research.data import (
    load_config, project_root, load_frame, save_frame, split_three_way, make_target,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]
dates = exp["dates"]

import pandas as pd
from spy_alpha_research.features import build_feature_frame, assert_feature_target_alignment
from spy_alpha_research.preprocessing import make_scaler
ohlcv = load_frame(PROC / "spy_ohlcv.parquet")
ret = load_frame(PROC / "spy_returns.parquet").iloc[:, 0]


In [ ]:
feats = build_feature_frame(
    ohlcv, ret,
    return_lags=exp["features"]["return_lags"],
    momentum_windows=exp["features"]["momentum_windows"],
    vol_windows=exp["features"]["vol_windows"],
    vol_of_vol_window=exp["features"]["vol_of_vol_window"],
    volume_windows=exp["features"]["volume_windows"],
    include_calendar=exp["features"]["include_calendar"],
)
target = make_target(ret, exp["target"]["horizon"])
assert_feature_target_alignment(feats, target, returns=ret)

train_idx, sel_idx, final_idx = split_three_way(
    feats.index,
    train_end=dates["train_end"],
    selection_start=dates["selection_start"],
    selection_end=dates["selection_end"],
    final_start=dates["final_start"],
    end=dates["end"],
)
scaler = make_scaler(
    exp["scaling"]["method"], exp["scaling"]["min_periods"], exp["scaling"]["rolling_window"]
)
X_train = scaler.fit_transform(feats.loc[train_idx])
X_sel = scaler.transform(feats.loc[sel_idx])
X_final = scaler.transform(feats.loc[final_idx])
save_frame(feats, PROC / "features_raw.parquet")
save_frame(X_train, PROC / "features_train_scaled.parquet")
save_frame(X_sel, PROC / "features_selection_scaled.parquet")
save_frame(X_final, PROC / "features_final_scaled.parquet")
save_frame(pd.concat([X_sel, X_final]).sort_index(), PROC / "features_test_scaled.parquet")
len(train_idx), len(sel_idx), len(final_idx), list(feats.columns)[:8]
